Welcome to Axiomatic Lab Kraken API function for Python

Sign up for Kraken here (Referral Link): https://proinvite.kraken.com/9f1e/2fth6ku2

Instructions on how to get started: https://axiomaticlab.com/portfolio/blog/guide-to-kraken-api-for-python

Kraken API: https://docs.kraken.com/institutional/api-reference/prime-rest
Kraken API Docs: https://docs.kraken.com/exchange/guides/rest/authentication#setting-the-api-key-parameter

In [ ]:
import sys, time, base64, hashlib, hmac, urllib.request, json, requests

def get_kraken_signature(urlpath, data, secret):

    postdata = urllib.parse.urlencode(data)
    encoded = (str(data['nonce']) + postdata).encode()
    message = urlpath.encode() + hashlib.sha256(encoded).digest()

    mac = hmac.new(base64.b64decode(secret), message, hashlib.sha512)
    sigdigest = base64.b64encode(mac.digest())
    return sigdigest.decode()

def kraken_api_functions(request_method, user_api_key=None, user_secret_key=None,):
    api_public = {"Time", "Assets", "AssetPairs", "Ticker", "OHLC", "Depth", "Trades", "Spread", "SystemStatus"}
    api_private = {"Balance", "BalanceEx", "TradeBalance", "OpenOrders", "ClosedOrders", "QueryOrders", "TradesHistory", "QueryTrades", "OpenPositions", "Ledgers", "QueryLedgers", "TradeVolume", "AddExport", "ExportStatus", "RetrieveExport", "RemoveExport", "GetWebSocketsToken", "CreateSubaccount", "AccountTransfer"}
    api_trading = {"AddOrder", "AddOrderBatch", "EditOrder", "CancelOrder", "CancelOrderBatch", "CancelAll", "CancelAllOrdersAfter"}
    api_funding = {"DepositMethods", "DepositAddresses", "DepositStatus", "WithdrawInfo", "Withdraw", "WithdrawStatus", "WithdrawCancel", "WalletTransfer"}
    api_staking = {"Earn/Strategies", "Earn/Allocations", "Earn/Allocate", "Earn/Deallocate", "Earn/AllocateStatus", "Earn/DeallocateStatus", "Staking/Assets", "Staking/Balance", "Stake", "Unstake", "Staking/Pending", "Staking/Transactions"}

    api_domain = "https://api.kraken.com"
    api_data = ""

    output_format = 0
    payload = 0
    headers = 0

    ### start

    split_request_metho = str(request_method).split(" ")

    if len(split_request_metho) < 2:
        api_method = split_request_metho[0]
    elif len(split_request_metho) <= 2:
        api_method = split_request_metho[1]
    else:
        api_method = split_request_metho[1]
        for count in range(2, len(split_request_metho)):
            if split_request_metho[count] == '-pretty':
                output_format = 1
                continue
            if count == 2:
                api_data = split_request_metho[count]
            else:
                api_data = api_data + "&" + split_request_metho[count]
    if api_method in api_private or api_method in api_trading or api_method in api_funding or api_method in api_staking:
        if user_api_key == None:
                    return {"status":"0", "response":"API public key and API private (secret) key must be in plain text files called API_Public_Key and API_Private_Key"}
        
        api_path = "/0/private/"
        api_nonce = str(int(time.time()*1000))
        payload = {"nonce": api_nonce,}
        temp_url = api_path + api_method

        signature = get_kraken_signature(temp_url, payload, user_secret_key)
        headers = {
                "API-Key":user_api_key,
                "API-Sign":signature.strip(),
            }
        
        api_request_url = api_domain + api_path + api_method
        

    elif api_method in api_public:
        api_path = "/0/public/"
        api_request_url = api_domain + api_path + api_method + '?' + api_data
        
    else:
        return {"status":"0", "response":["Usage: %s method [parameters]" % split_request_metho[0], "Example: %s OHLC pair=xbtusd interval=1440" % split_request_metho[0]]}

    try:
        if payload == 0 or headers == 0:
            api_reply = requests.post(api_request_url)
        else:
            print("yes")
            api_reply = requests.post(api_request_url, data={"nonce": api_nonce,}, headers=headers)
    except Exception as error:
        return {"status":"0", "response":["API call failed (%s)" % error]}

    api_reply = api_reply.text
        
    if '"error":[]' in api_reply:
        if output_format == 0:
            return {"status":"0.5", "response":api_reply}
        else:
            return {"status":"0.5", "response":json.dumps(json.loads(api_reply), indent = 4)}

    else:
        if output_format == 0:
            return {"status":"1", "response":api_reply}
        else:
            return {"status":"1", "response":json.dumps(json.loads(api_reply), indent = 4)}



In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
test_request = "Balance"
my_api_key = os.getenv("NEW_TEST_API_KEY")
my_secret_key = os.getenv("NEW_TEST_PRIVATE_KEY")
temp_repsonse = kraken_api_functions(test_request, my_api_key, my_secret_key)

print(temp_repsonse)

In [ ]:
test_request = "OHLC OHLC pair=xbtusd interval=1440"
temp_repsonse = kraken_api_functions(test_request, )
#print(temp_repsonse)

In [6]:
import pandas as pd

temp_repsonse_data = json.loads(temp_repsonse['response'])
temp_repsonse_data = temp_repsonse_data['result']

data_cols = ["time", "open", "high", "low", "close", "vwap", "vol", "num"]

temp_data_df = pd.DataFrame(temp_repsonse_data['XXBTZUSD'], columns=data_cols)
temp_data_df['time'] = pd.to_datetime(temp_data_df['time'], unit='s')
#temp_data_df['time'] = temp_data_df['time']/1000

print(temp_data_df.head(5))

        time     open     high      low    close     vwap            vol  \
0 2024-11-12  88737.5  89920.0  85136.0  88087.3  87946.7  6817.99545766   
1 2024-11-13  88087.3  93435.0  86258.0  90490.9  90290.9  7626.54566803   
2 2024-11-14  90496.8  91780.0  86621.9  87313.2  89304.9  6252.08697742   
3 2024-11-15  87314.4  91850.0  87135.0  91062.0  89507.4  3879.47146166   
4 2024-11-16  91062.0  91750.0  90149.0  90555.1  91055.2  1466.32281088   

     num  
0  85465  
1  77561  
2  61715  
3  48562  
4  32817  
